# Derivative Pricing - Black-Scholes Sequential

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import datetime
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from DerivModel import train_model, FeedForwardNetwork, black_scholes
from DerivMetrics import evaluate_metrics, compute_metrics 
from DerivPlots import scatter_plot, plot_errors
import json
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from tqdm.notebook import tqdm, trange

## 20-day Historical Vol

### Load and Prepare the dataframe

In [ ]:
in_file = 'SPXOptions.csv'
dfopt = pd.read_csv(in_file)

In [ ]:
dfopt

In [ ]:
device = torch.device("cuda:0") if torch.cuda.is_available() else torch.device("cpu")
print("Device:", device)

In [ ]:
dfopt['type'] = dfopt['type'].replace({'call': 1, 'put': 0})

In [ ]:
# Sort the DataFrame by 'date_column' in increasing order
df_sorted = dfopt.sort_values(by='quote_date')

# Reset the index if needed
df_sorted = df_sorted.reset_index(drop=True)

In [ ]:
# Define the percentage to split
x_percent = 80  # Change this to the desired percentage

# Calculate the number of rows to take for the top x% and the remainder
total_rows = len(df_sorted)
top_x_percent = int((x_percent / 100) * total_rows)
remainder = total_rows - top_x_percent

# Split the DataFrame
df_top_x_percent = df_sorted.head(top_x_percent)
df_remainder = df_sorted.tail(remainder)

# Reset the index for both DataFrames if needed
df_top_x_percent = df_top_x_percent.reset_index(drop=True)
df_remainder = df_remainder.reset_index(drop=True)

In [ ]:
#Separate into X and y
X_train = df_top_x_percent[['type', 'strike', '20D_HV_x', 'maturity', 'rate', 'underlying']]
y_train = df_top_x_percent['mid']

In [ ]:
#Separate into X and y
X_test = df_remainder[['type', 'strike', '20D_HV_x', 'maturity', 'rate', 'underlying']]
y_test = df_remainder['mid']

In [ ]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [ ]:
torch_tensor = torch.from_numpy(X_train).float().to(device)
y_tensor = torch.from_numpy(y_train.to_numpy()).float().to(device)
dataset = TensorDataset(torch_tensor, y_tensor)
batch_size = 512
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=False, drop_last=True)

test_torch_tensor = torch.from_numpy(X_test).float().to(device)
test_y_tensor = torch.from_numpy(y_test.to_numpy().copy()).float().to(device)
test_dataset = TensorDataset(test_torch_tensor, test_y_tensor)
test_data_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

### Build and Train model

In [ ]:
epochs = 50
no_layers = 5
no_neurons = 4096
lr = 0.001

In [ ]:
model = FeedForwardNetwork(input_size=6, 
                           num_hidden_layers=no_layers, 
                           neurons_per_layer=no_neurons).to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
loss_fn = torch.nn.MSELoss()

In [ ]:
train_errors, test_errors = train_model(model, data_loader, test_data_loader, 
                                        loss_fn, optimizer, epochs)

In [ ]:
plot_file = 'bs_hist20_SE.png'
plot_errors(train_errors, test_errors, plot_file)

In [ ]:
metrics, all_results, all_targets = evaluate_metrics(test_data_loader, model)

In [ ]:
metrics

In [ ]:
metric_file = 'bsmetrics_hist20_SE.json'
with open(metric_file, 'w') as json_file:
    json.dump(metrics, json_file)

In [ ]:
scatter_filename = 'bsscatter_hist20_SE.png'
scatter_plot(all_results, all_targets, scatter_filename)